# Visualizzare una trattografia
Potete tranquillamente visualizzare le trattografie in fsleyes, farlo in python vi dà più performance e più flessibilità nella visualizzazione.

In [ ]:
import nibabel as nib
import numpy as np
import pyvista as pv

# Load the Data
file_path = "HCP101309/tracks_20k_sift2.tck"
tck_data = nib.streamlines.load(file_path)

In [ ]:
# Extract the streamlines
# Note: Tractograms can have millions of fibers. For testing visualization, 
# it is highly recommended to plot a subset first so you don't freeze your computer!
streamlines = tck_data.streamlines[:1000] # Just taking the first 1000

# --- 2. Convert to PyVista PolyData ---
# PyVista needs all points in one giant array
points = np.vstack(streamlines)

# PyVista also needs a "connectivity" array telling it how points connect into lines.
# Format: [N, p0, p1... pN, M, q0, q1... qM] where N, M are number of points in the line.
lines = []
current_index = 0

for sl in streamlines:
    n_pts = len(sl)
    lines.append(n_pts) # First element is the number of points in this specific line
    lines.extend(range(current_index, current_index + n_pts)) # Followed by the point indices
    current_index += n_pts

lines = np.array(lines)

# Create the PyVista PolyData object
mesh = pv.PolyData(points, lines=lines)

# --- 3. Render the Scene ---
plotter = pv.Plotter()

# Add the tractography mesh to the scene
plotter.add_mesh(
    mesh, 
    color="lightblue", 
    line_width=2.0, 
    opacity=0.8,
    render_lines_as_tubes=True # Makes the lines look 3D rather than flat pixels
)

# Optional: Add a bounding box to help with 3D perspective
plotter.add_bounding_box()

# Open the interactive window!
plotter.show()

## Plot with colors
Typically, tractographies are plotted in colors to make their direction more clear, let's do that

In [ ]:
# Extract the streamlines
# Note: Tractograms can have millions of fibers. For testing visualization, 
# it is highly recommended to plot a subset first so you don't freeze your computer!
streamlines = tck_data.streamlines[:1000] # let's push it a bit higher

# --- 2. Convert to PyVista PolyData ---
# PyVista needs all points in one giant array
points = np.vstack(streamlines)

# PyVista also needs a "connectivity" array telling it how points connect into lines.
# Format: [N, p0, p1... pN, M, q0, q1... qM] where N, M are number of points in the line.
lines = []
current_index = 0

for sl in streamlines:
    n_pts = len(sl)
    lines.append(n_pts) # First element is the number of points in this specific line
    lines.extend(range(current_index, current_index + n_pts)) # Followed by the point indices
    current_index += n_pts

lines = np.array(lines)

# Create the PyVista PolyData object
mesh = pv.PolyData(points, lines=lines)

# Calculate the direction (tangent) vector between consecutive points
directions = np.zeros_like(points)

# A simple way to approximate the direction vector for each point
directions[:-1] = points[1:] - points[:-1]
directions[-1] = directions[-2] # Handle the very last point

# Normalize the vectors (so RGB values stay between 0 and 1)
norms = np.linalg.norm(directions, axis=1)
norms[norms == 0] = 1 # Prevent division by zero
directions = np.abs(directions / norms[:, np.newaxis]) # Absolute value for colors

# Assign these colors to the mesh points
mesh.point_data["Direction_RGB"] = directions

# Plot it!
plotter = pv.Plotter()
plotter.add_mesh(
    mesh, 
    scalars="Direction_RGB", 
    rgb=True,               # Tell PyVista to interpret the data as RGB colors
    line_width=2.0, 
    render_lines_as_tubes=True
)
plotter.show()

# Matrici strutturali
Tipicamente, le trattografie vengono usate per capire quanto sono interconnesse le varie regioni del cervello. Il processo di calcolo può essere un po' intensivo, quindi qui esploreremo direttamente il risultato.

In [ ]:
import numpy as np
from nilearn import plotting
import matplotlib.pyplot as plt

## Atlante 100
Questo atlante suddivide il cervello in 100 regioni corticali (50 a destra e 50 a sinistra) e 106 regioni sottocorticali.

I file che contengono queste matrici sono semplici file di testo, potete aprirli ed esplorarli con un qualsiasi editor.

### Labels

In [ ]:
# a (pretty bad) way of reading this file
with open('structural_matrices/labels_100.txt', 'r') as f:
    labels = f.readlines()

# elimina lo \n finale
for i in range(len(labels)):
    labels[i] = labels[i].strip()

# converti in array per poter usare il "fancy indexing"
labels = np.array(labels)

labels

### Weights
Una matrice che ci dice quanto intensamente sono interconnesse le regioni. Conta la quantità di "tratti" che collegano due regioni, e rinormalizza il tutto.

In [ ]:
# a (pretty bad) way of reading a matrix
with open('structural_matrices/weights_100.txt', 'r') as f:
    weights_file = f.readlines()

weights = np.zeros((len(weights_file), len(weights_file)))
for i, vec in enumerate(weights_file):
    vec = vec.strip().split(' ')
    for j, val in enumerate(vec):
        weights[i,j] = float(val)
weights

In [ ]:
# Plot the matrix

plt.figure(figsize=(8, 6))

# Choose a colormap and set the color for NaN (our zeros)
cmap = plt.cm.viridis
cmap.set_bad(color='lightgray')

# Plot using LogNorm
plt.imshow(
    weights, 
    cmap=cmap, 
    interpolation='none'
)

plt.colorbar(label='Weight')
plt.title("Structural Connectivity (Log Scale)")
plt.show()

Non si vede granché, proviamo a fare un istogramma:

In [ ]:
plt.hist(weights.flatten(), log = True)

La stragrande maggioranza dei valori sono vicini allo zero (nota la scala logaritmica). Plottiamo la matrice imponendo una scala logaritmica sulla colorbar:

In [ ]:
from matplotlib.colors import LogNorm

plt.figure(figsize=(8, 6))

# Find the minimum non-zero value to set as the bottom of our log scale
vmin = weights[weights > 0].min()
vmax = weights.max()

# Replace 0s with NaNs just for plotting, so they get the 'bad' color
weights_plot = np.where(weights == 0, vmin, weights)

# Choose a colormap and set the color for NaN (our zeros)
cmap = plt.cm.viridis
cmap.set_bad(color='lightgray')

# Plot using LogNorm
plt.imshow(
    weights_plot, 
    cmap=cmap, 
    norm=LogNorm(vmin=vmin, vmax=vmax), 
    interpolation='none'
)

plt.colorbar(label='Log(Weight)')
plt.title("Structural Connectivity (Log Scale)")
plt.show()

### Distances
Questa matrice contiene la lunghezza (in mm) di ogni tratto.

In [ ]:
# a (better) way of reading a matrix
distances = np.genfromtxt('structural_matrices/distances_100.txt', dtype = float, delimiter = ' ')
distances

In [ ]:
# plot this matrix with labels on the side

fig, ax = plt.subplots(figsize=(25, 25), tight_layout = True) 

plotting.plot_matrix(
    distances,
    labels=labels,
    axes=ax,
    colorbar=True,
    cmap='hot',
    title='Fiber length between regions'
)

ax.tick_params(axis='x', labelsize=5, labelrotation=90) # Rotate X labels so they don't crash
ax.tick_params(axis='y', labelsize=5)
ax.set_title('Fiber length between regions (mm)', fontdict={'fontsize':20})

plt.savefig('distances_matrix_100.png', dpi = 300)
plt.show()

## YOUR TURN: Atlante 1000
Questo atlante suddivide il cervello in 1000 regioni corticali (500 a destra e 500 a sinistra) e 106 regioni sottocorticali.

Replica le analisi di sopra con questo atlante. Cosa ti aspetti di diverso?

### Labels

In [ ]:
# a (better) way of reading this file
labels = np.genfromtxt('structural_matrices/labels_1000.txt', dtype = str, delimiter = '')
labels

### Weights

In [ ]:
### YOUR CODE

### Distances

In [ ]:
### YOUR CODE

## Interazione tra la distanza e il peso di connessione
Come analisi di base, proviamo a capire la relazione tra distanza e peso di connessione tra due regioni nel cervello.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

weights = np.genfromtxt('structural_matrices/weights_100.txt', dtype = float, delimiter = ' ')
distances = np.genfromtxt('structural_matrices/distances_100.txt', dtype = float, delimiter = ' ')

In [ ]:
# Get the indices for the upper triangle of the matrix (ignoring the diagonal)
row_idx, col_idx = np.triu_indices(distances.shape[0], k=1)

# Extract the actual values using those indices
dist_vals = distances[row_idx, col_idx]
weight_vals = weights[row_idx, col_idx]

# Filter out absolute zeros (regions that have NO tracked connection)
nonzero_mask = weight_vals > 0
dist_vals_filtered = dist_vals[nonzero_mask]
weight_vals_filtered = weight_vals[nonzero_mask]

# Create a side-by-side figure (1 row, 2 columns)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ---------------------------------------------------------
# Plot 1: Linear Scale (Left Side)
# ---------------------------------------------------------
ax1.scatter(dist_vals_filtered, weight_vals_filtered, alpha=0.2, color='teal')
ax1.set_title('Wiring Cost: Distance vs. Weight\n(Linear Scale)')
ax1.set_xlabel('Fiber Length (mm)')
ax1.set_ylabel('Connection Weight')
ax1.grid(True, alpha=0.3)

# ---------------------------------------------------------
# Plot 2: Log Scale (Right Side)
# ---------------------------------------------------------
ax2.scatter(dist_vals_filtered, weight_vals_filtered, alpha=0.2, color='teal')
ax2.set_title('Wiring Cost: Distance vs. Weight\n(Log Scale)')
ax2.set_xlabel('Fiber Length (mm)')
ax2.set_ylabel('Connection Weight (Log)')
ax2.set_yscale('log') # This turns on the log scale for this specific axis
ax2.grid(True, alpha=0.3)

# tight_layout() prevents the labels of the left plot from overlapping the right plot
plt.tight_layout()
plt.show()